In [1]:
import os
import pandas as pd
from datetime import datetime
import json
import gc

folder_path_demanddetails = '/home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/'

# read active properties & needed columns
property_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/abohar/eg_pt_property.csv',
    usecols=['id', 'propertyid', 'tenantid', 'createdtime', 'additionaldetails', 'ownershipcategory', 'status', 'usagecategory']
)
property_df = property_df[property_df['status'] == 'ACTIVE'].copy()

# read units
# unit_df = pd.read_csv(
#     '/home/prerna/Punjab/punjab-data-prod-analysis/srihargobindpur/eg_pt_unit.csv',
#     usecols=['propertyid', 'occupancytype']
# )



# read demand
demand_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/abohar/egbs_demand_v1.csv',
    dtype={"consumercode": str},
    low_memory=False,
    usecols=['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
)
demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()


# read demand details (memory‑efficient, in chunks)
all_chunks = []
needed_cols = ['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
for filename in os.listdir(folder_path_demanddetails):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path_demanddetails, filename)
        print(f'Loading: {file_path}')
        chunk = pd.read_csv(file_path, usecols=needed_cols)
        all_chunks.append(chunk)
demand_details_df = pd.concat(all_chunks, ignore_index=True)
del all_chunks; gc.collect()

print("✅ Loaded data")

Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_4.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_34.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_79.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_38.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_6.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_60.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_44.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_83.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_89.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/abohar/output_demand_details/output_17.csv
Loading: /home/prerna/

In [2]:
print(len(property_df))         # number of rows in properties
# print(len(unit_df))             # number of rows in units
print(len(demand_df))   # number of rows in demand details
print(len(demand_details_df))   # number of rows in demand details

43158
178240
5217011


In [3]:
# join demand and demand details
joined_demand = demand_df.merge(demand_details_df, left_on='id', right_on='demandid', how='left', suffixes=('_demand', '_detail'))
print(joined_demand['id'].nunique())
del demand_details_df, demand_df; gc.collect()
joined_demand.head()

178240


,id,consumercode,businessservice,taxperiodfrom,taxperiodto,status,demandid,taxheadcode,taxamount,collectionamount
0,18839,PT-601-010237,PT,1459468800000,1491004799000,ACTIVE,18839,PT_TIME_REBATE,0.00,0.00
1,18839,PT-601-010237,PT,1459468800000,1491004799000,ACTIVE,18839,PT_TIME_PENALTY,158.16,158.16
2,18839,PT-601-010237,PT,1459468800000,1491004799000,ACTIVE,18839,PT_TIME_INTEREST,279.62,279.62
3,18839,PT-601-010237,PT,1459468800000,1491004799000,ACTIVE,18839,PT_ADHOC_REBATE,-51.00,-51.00
4,18839,PT-601-010237,PT,1459468800000,1491004799000,ACTIVE,18839,PT_ROUNDOFF,-0.38,-0.38


In [4]:
import pytz

# Correct: parse as datetime from milliseconds since epoch
joined_demand['taxperiodfrom'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)
joined_demand['taxperiodto'] = pd.to_datetime(joined_demand['taxperiodto'], unit='ms', utc=True)

# Convert to IST (Asia/Kolkata)
ist = pytz.timezone('Asia/Kolkata')
joined_demand['taxperiodfrom'] = joined_demand['taxperiodfrom'].dt.tz_convert(ist)
joined_demand['taxperiodto'] = joined_demand['taxperiodto'].dt.tz_convert(ist)

# Financial year calculation
def get_fy(date):
    if date.month >= 4:
        fy_start = date.year
        fy_end = date.year + 1
    else:
        fy_start = date.year - 1
        fy_end = date.year
    return f"{fy_start}-{str(fy_end)[-2:]}"

joined_demand['fy'] = joined_demand['taxperiodfrom'].apply(get_fy)

# Group by consumercode
result = joined_demand.groupby('consumercode')['fy'].agg(['min', 'max']).reset_index()
result.rename(columns={'min': 'earliest_fy', 'max': 'latest_fy'}, inplace=True)

print(result)

        consumercode earliest_fy latest_fy
0      PT-601-010237     2016-17   2024-25
1      PT-601-018942     2014-15   2024-25
2      PT-601-019040     2014-15   2024-25
3      PT-601-019057     2014-15   2024-25
4      PT-601-019216     2014-15   2025-26
...              ...         ...       ...
46328  PT-601-999774     2017-18   2024-25
46329  PT-601-999830     2014-15   2025-26
46330  PT-601-999883     2014-15   2024-25
46331  PT-601-999888     2016-17   2024-25
46332  PT-601-999942     2020-21   2024-25

[46333 rows x 3 columns]


In [5]:
# Merge latest_fy onto joined_demand by consumercode
joined = joined_demand.merge(
    result[['consumercode', 'latest_fy']],
    on='consumercode',
    how='left'
)

# Filter only latest FY
latest_demand = joined[joined['fy'] == joined['latest_fy']]

# Pivot taxheadcode values into separate columns
pivoted = latest_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
# PT_TAX + PT_CANCER_CESS + PT_FIRE_CESS + PT_ROUNDOFF - (PT_OWNER_EXEMPTION + PT_UNIT_USAGE_EXEMPTION)
pivoted['latest_fy_taxamount'] = (
    pivoted.get('PT_TAX', 0) +
    pivoted.get('PT_CANCER_CESS', 0) +
    pivoted.get('PT_FIRE_CESS', 0) +
    pivoted.get('PT_ROUNDOFF', 0) -
    ( pivoted.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Merge back into result
result = result.merge(
    pivoted[['consumercode', 'latest_fy_taxamount']],
    on='consumercode',
    how='left'
)

print(result.head())


    consumercode earliest_fy latest_fy  latest_fy_taxamount
0  PT-601-010237     2016-17   2024-25               980.12
1  PT-601-018942     2014-15   2024-25               247.27
2  PT-601-019040     2014-15   2024-25               261.68
3  PT-601-019057     2014-15   2024-25              3220.06
4  PT-601-019216     2014-15   2025-26               723.64


In [6]:
# Calculating the tax amount (demand) of current year using formula
target_fy = "2025-26"
current_fy_demand = joined_demand[joined_demand['fy'] == target_fy]

# Pivot taxheadcode values into separate columns
pivoted_current = current_fy_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
pivoted_current['current_fy_taxamount'] = (
    pivoted_current.get('PT_TAX', 0) +
    pivoted_current.get('PT_CANCER_CESS', 0) +
    pivoted_current.get('PT_FIRE_CESS', 0) +
    pivoted_current.get('PT_ROUNDOFF', 0) -
    ( pivoted.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Keep only required cols
pivoted_current = pivoted_current[['consumercode', 'current_fy_taxamount']]

# Ensure all consumercodes are present
all_consumercodes = pd.DataFrame(joined_demand['consumercode'].unique(), columns=['consumercode'])
final = all_consumercodes.merge(pivoted_current, on='consumercode', how='left')
final['current_fy_taxamount'] = final['current_fy_taxamount'].fillna(0)

# Merge into result
result = result.merge(final, on='consumercode', how='left')
result['current_fy_taxamount'] = result['current_fy_taxamount'].fillna(0)

print(result.head())


    consumercode earliest_fy latest_fy  latest_fy_taxamount  \
0  PT-601-010237     2016-17   2024-25               980.12   
1  PT-601-018942     2014-15   2024-25               247.27   
2  PT-601-019040     2014-15   2024-25               261.68   
3  PT-601-019057     2014-15   2024-25              3220.06   
4  PT-601-019216     2014-15   2025-26               723.64   

   current_fy_taxamount  
0                  0.00  
1                  0.00  
2                  0.00  
3                  0.00  
4                723.64  


In [7]:
property_result_merged = property_df.merge(
    result,
    left_on='propertyid',
    right_on='consumercode',
    how='left'
)

print(property_result_merged)

                                         id      propertyid   tenantid  \
0      7eafe3da-ad69-441a-a982-899a80561e9b  PT-601-2154555  pb.abohar   
1      d4572a6e-e993-4245-a7cd-7ff83d324e51  PT-601-2154556  pb.abohar   
2      2fb0fe75-4e5b-43d2-b681-7b7b9e6251be  PT-601-2110140  pb.abohar   
3      86fa4b2d-3a27-4b7c-b5bb-c822cb1a6a80  PT-601-2154557  pb.abohar   
4      813b3aa9-c87b-4d55-994f-7e8d951f03c3  PT-601-2154559  pb.abohar   
...                                     ...             ...        ...   
43153  7860e170-a4e3-4fe6-8cb8-0acc67082996   PT-601-098718  pb.abohar   
43154  1b54e1a5-8cfb-409a-8f2e-0dd235677136  PT-601-1030985  pb.abohar   
43155  9da1008b-1f39-4479-9a96-169fc4cea80c   PT-601-774270  pb.abohar   
43156  cc5b2b79-4741-4734-bc0e-39ab322423a8   PT-601-845206  pb.abohar   
43157  bf65027f-5a50-48b0-a313-bc6826a18b20  PT-601-1154939  pb.abohar   

       status          ownershipcategory              usagecategory  \
0      ACTIVE     INDIVIDUAL.SINGLEOWNER

In [8]:
property_result_merged.to_csv('Punjab_Data_Analysis_abohar_final.csv', index=False)